In [1]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.0 which is incompatible.


In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [3]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """This function fetches the currency conversion factor between given base currency & target currency"""
  url = f'https://v6.exchangerate-api.com/v6/api/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """Given a currency conversion rate and base currency value, this function calculates the target currency value"""
  return base_currency_value*conversion_rate


In [4]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1761264001,
 'time_last_update_utc': 'Fri, 24 Oct 2025 00:00:01 +0000',
 'time_next_update_unix': 1761350401,
 'time_next_update_utc': 'Sat, 25 Oct 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 87.8574}

In [5]:
convert.invoke({'base_currency_value': 50, 'conversion_rate': 86.01})

4300.5

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

llm = ChatOpenAI(
    openai_api_key="api key",
    openai_api_base="https://openrouter.ai/api/v1",
    model="google/gemini-2.5-flash"
)

In [7]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [8]:
messages = [HumanMessage('What is the conversion rate of USD to INR, and can you convert 50 USD to INR')]

In [9]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 143, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash', 'system_fingerprint': None, 'id': 'gen-1761306792-RcNHYEdsojoEMqUSGJTq', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--f6b8c9ff-f6f8-49d6-919b-87efbc202c9f-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'tool_0_get_conversion_factor_PNI6snwAmx0rN6T52GZU', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 50}, 'id': 'tool_1_convert_fic1Y7x9FqHFDTSAKtIL', 'type': 'tool_call'}], usage_metadata={'input_tokens': 143, 'output

In [10]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'tool_0_get_conversion_factor_AzE5e5W20TYQgM4mrFko',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 50},
  'id': 'tool_1_convert_VcsfqtollE6ZuKjjXlhL',
  'type': 'tool_call'}]

In [12]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'tool_0_get_conversion_factor_yJ2RhTZKzIprrqLOH0AZ',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 50},
  'id': 'tool_1_convert_KszYmZdinrqeu5agKVW6',
  'type': 'tool_call'}]

In [13]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'tool_0_get_conversion_factor_ZCu6Rj72LtHIl0QHiZwP',
  'type': 'tool_call'}]

In [14]:
ai_message = llm_with_tools.invoke(messages)

In [15]:
messages.append(ai_message)

In [16]:
messages

[HumanMessage(content='What is the conversion rate of USD to INR, and can you convert 50 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 143, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash', 'system_fingerprint': None, 'id': 'gen-1761306850-G7GGhEWg6Zr60kkSkVqQ', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--1aace3c4-10f8-412c-b2e9-626676aabc48-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'tool_0_get_conversion_factor_j1kXCHQWz0jcvkiJzBgE', 'type': 'tool_call'}, {'name': 'convert', '

In [17]:
import json

for tool_call in ai_message.tool_calls:
  #execute the first tool and get the conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 =get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [18]:
messages

[HumanMessage(content='What is the conversion rate of USD to INR, and can you convert 50 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 143, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash', 'system_fingerprint': None, 'id': 'gen-1761306850-G7GGhEWg6Zr60kkSkVqQ', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--1aace3c4-10f8-412c-b2e9-626676aabc48-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'tool_0_get_conversion_factor_j1kXCHQWz0jcvkiJzBgE', 'type': 'tool_call'}, {'name': 'convert', '

In [19]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion rate from USD to INR is 87.8574.\n50 USD is equal to 4392.87 INR.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 459, 'total_tokens': 492, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash', 'system_fingerprint': None, 'id': 'gen-1761306864-zP1qOJDVgnThH4HAQaIH', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--77bcb6de-4cd1-4798-9640-8422b2524f47-0', usage_metadata={'input_tokens': 459, 'output_tokens': 33, 'total_tokens': 492, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}})